# MyDigitalTwin — Spotify
**Notebook — Ingestion incrémentale → Warehouse Delta Lake**

Sources (`data/processed/SPOTIFY/`) :
- `account/StreamingHistory_music_*.json` — historique récent, ~1 an (format court)
- `extended/Streaming_History_Audio_*.json` — historique complet depuis 2020 (format riche)
- `account/YourLibrary.json` — titres likés (identité musicale forte)
- `account/Playlist1.json` — playlists personnelles
- `account/SearchQueries.json` — recherches Spotify

Outputs (warehouse Delta Lake) :
- `spotify_streams` — écoutes nettoyées (>30s), fusion Extended+Account, déduplication
- `spotify_liked_songs` — titres likés avec `trackUri` (pour graphe topologique)
- `spotify_playlists` — tracks de toutes les playlists
- `spotify_searches` — requêtes de recherche

## Stratégie d'ingestion incrémentale
| Source | Couverture | Schéma |
|---|---|---|
| Extended | 2020-11-21 → 2025-05-03 | Riche (`trackUri`, `skipped`, `shuffle`) |
| Account  | 2025-03-29 → 2026-03-30 | Simple (`endTime`, `msPlayed`) |

**Principe** : union des deux sources → déduplication par clé `(artistName, trackName, minute)` en favorisant Extended → écriture Delta `overwrite` (idempotent).

Lors des prochains exports, les nouveaux fichiers s'accumulent dans `processed/` (parser OVERWRITE=False).  
Il suffit de relancer ce notebook pour intégrer les nouveaux événements sans doublons.

## 0. Initialisation Spark

In [1]:
import sys, os
_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)
from config import build_spark_session, PROCESSED_DATA, WAREHOUSE

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType
import glob

spark = build_spark_session("MyDigitalTwin - Spotify")
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"PROCESSED_DATA : {PROCESSED_DATA}")
print(f"WAREHOUSE      : {WAREHOUSE}")

Spark version : 3.5.5
PROCESSED_DATA : /opt/spark/data/processed
WAREHOUSE      : /opt/spark/data/warehouse


26/04/27 13:07:16 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Chemins & Configuration

In [2]:
SPOTIFY_ACCOUNT  = os.path.join(PROCESSED_DATA, "SPOTIFY", "account")
SPOTIFY_EXTENDED = os.path.join(PROCESSED_DATA, "SPOTIFY", "extended")

LIBRARY_PATH      = os.path.join(SPOTIFY_ACCOUNT, "YourLibrary.json")
PLAYLIST_PATH     = os.path.join(SPOTIFY_ACCOUNT, "Playlist1.json")
SEARCH_PATH       = os.path.join(SPOTIFY_ACCOUNT, "SearchQueries.json")

OUT_STREAMS       = os.path.join(WAREHOUSE, "spotify_streams")
OUT_LIKED_SONGS   = os.path.join(WAREHOUSE, "spotify_liked_songs")
OUT_PLAYLISTS     = os.path.join(WAREHOUSE, "spotify_playlists")
OUT_SEARCHES      = os.path.join(WAREHOUSE, "spotify_searches")

# Lister les fichiers disponibles
extended_files = sorted(glob.glob(os.path.join(SPOTIFY_EXTENDED, "Streaming_History_Audio_*.json")))
account_files  = sorted(glob.glob(os.path.join(SPOTIFY_ACCOUNT,  "StreamingHistory_music_*.json")))

print(f"Extended History : {len(extended_files)} fichiers")
for f in extended_files:
    print(f"  {os.path.basename(f)}")
print(f"\nAccount Data     : {len(account_files)} fichiers")
for f in account_files:
    print(f"  {os.path.basename(f)}")

Extended History : 19 fichiers
  Streaming_History_Audio_2020-2022_0.json
  Streaming_History_Audio_2022-2023_4.json
  Streaming_History_Audio_2022_1.json
  Streaming_History_Audio_2022_2.json
  Streaming_History_Audio_2022_3.json
  Streaming_History_Audio_2023-2024_10.json
  Streaming_History_Audio_2023_5.json
  Streaming_History_Audio_2023_6.json
  Streaming_History_Audio_2023_7.json
  Streaming_History_Audio_2023_8.json
  Streaming_History_Audio_2023_9.json
  Streaming_History_Audio_2024-2025_16.json
  Streaming_History_Audio_2024_11.json
  Streaming_History_Audio_2024_12.json
  Streaming_History_Audio_2024_13.json
  Streaming_History_Audio_2024_14.json
  Streaming_History_Audio_2024_15.json
  Streaming_History_Audio_2025_17.json
  Streaming_History_Audio_2025_18.json

Account Data     : 10 fichiers
  StreamingHistory_music_0.json
  StreamingHistory_music_1.json
  StreamingHistory_music_2.json
  StreamingHistory_music_3.json
  StreamingHistory_music_4.json
  StreamingHistory_music_5

## 2. Extended Streaming History

Source principale : 19 fichiers JSON depuis 2020.  
Schéma riche : `ts`, `ms_played`, `master_metadata_*`, `spotify_track_uri`, `skipped`, `shuffle`.

In [3]:
df_ext_raw = spark.read \
    .option("multiLine", "true") \
    .json(extended_files)

print(f"Lignes brutes Extended : {df_ext_raw.count():,}")
print("Schema :")
df_ext_raw.printSchema()

# Normalisation vers schéma unifié
# ts format : "2020-11-21T12:47:13Z" (ISO 8601 UTC)
df_ext = df_ext_raw \
    .filter(F.col("master_metadata_track_name").isNotNull()) \
    .select(
        F.to_timestamp(F.col("ts"), "yyyy-MM-dd'T'HH:mm:ss'Z'").alias("listen_ts"),
        F.col("master_metadata_album_artist_name").alias("artistName"),
        F.col("master_metadata_track_name").alias("trackName"),
        F.col("ms_played").cast(LongType()).alias("msPlayed"),
        F.col("spotify_track_uri").alias("trackUri"),
        F.col("skipped").cast(BooleanType()).alias("skipped"),
        F.col("shuffle").cast(BooleanType()).alias("shuffle"),
        F.lit("extended").alias("_source"),
    ) \
    .filter(F.col("listen_ts").isNotNull())

print(f"\nLignes Extended normalisées (tracks uniquement) : {df_ext.count():,}")
df_ext.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier")
).show(truncate=False)

Lignes brutes Extended : 298,572
Schema :
root
 |-- audiobook_chapter_title: string (nullable = true)
 |-- audiobook_chapter_uri: string (nullable = true)
 |-- audiobook_title: string (nullable = true)
 |-- audiobook_uri: string (nullable = true)
 |-- conn_country: string (nullable = true)
 |-- episode_name: string (nullable = true)
 |-- episode_show_name: string (nullable = true)
 |-- incognito_mode: boolean (nullable = true)
 |-- ip_addr: string (nullable = true)
 |-- master_metadata_album_album_name: string (nullable = true)
 |-- master_metadata_album_artist_name: string (nullable = true)
 |-- master_metadata_track_name: string (nullable = true)
 |-- ms_played: long (nullable = true)
 |-- offline: boolean (nullable = true)
 |-- offline_timestamp: long (nullable = true)
 |-- platform: string (nullable = true)
 |-- reason_end: string (nullable = true)
 |-- reason_start: string (nullable = true)
 |-- shuffle: boolean (nullable = true)
 |-- skipped: boolean (nullable = true)
 |-- spotif


Lignes Extended normalisées (tracks uniquement) : 298,538


+-------------------+-------------------+
|premier            |dernier            |
+-------------------+-------------------+
|2020-11-21 12:47:13|2025-05-03 23:06:26|
+-------------------+-------------------+



## 3. Account Data Streaming History

Source complémentaire : couvre la période **2025-05-03 → 2026-03-30** non présente dans Extended.  
Schéma simple : `endTime` (minute), `artistName`, `trackName`, `msPlayed`.

In [4]:
df_acc_raw = spark.read \
    .option("multiLine", "true") \
    .json(account_files)

print(f"Lignes brutes Account Data : {df_acc_raw.count():,}")

# Normalisation vers le même schéma unifié
# endTime format : "2025-03-29 17:55" (UTC, précision minute)
df_acc = df_acc_raw \
    .filter(F.col("trackName").isNotNull()) \
    .select(
        F.to_timestamp(F.col("endTime"), "yyyy-MM-dd HH:mm").alias("listen_ts"),
        F.col("artistName"),
        F.col("trackName"),
        F.col("msPlayed").cast(LongType()),
        F.lit(None).cast(StringType()).alias("trackUri"),
        F.lit(None).cast(BooleanType()).alias("skipped"),
        F.lit(None).cast(BooleanType()).alias("shuffle"),
        F.lit("account").alias("_source"),
    ) \
    .filter(F.col("listen_ts").isNotNull())

print(f"Lignes Account normalisées : {df_acc.count():,}")
df_acc.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier")
).show(truncate=False)

Lignes brutes Account Data : 92,037
Lignes Account normalisées : 92,037
+-------------------+-------------------+
|premier            |dernier            |
+-------------------+-------------------+
|2024-05-04 21:22:00|2025-05-05 23:45:00|
+-------------------+-------------------+



## 4. Fusion & Déduplication (ingestion incrémentale)

**Clé de déduplication** : `(artistName, trackName, minute)` où `minute = date_trunc("minute", listen_ts)`.

En cas de doublon (même écoute présente dans Extended ET Account), on conserve la version **Extended** (schéma plus riche : `trackUri`, `skipped`, `shuffle`).  
Les écoutes uniques à Account (2025-05-03 → 2026-03-30) sont toutes conservées.

Ce mécanisme est **idempotent** : relancer le notebook avec les mêmes fichiers donne exactement le même résultat.

In [5]:
# Union des deux sources
df_all = df_ext.union(df_acc)

print(f"Total avant déduplication : {df_all.count():,}")
df_all.groupBy("_source").count().orderBy("_source").show()

# Colonnes helper pour la déduplication
# _dedup_min : clé à la minute (Extended = seconde, Account = minute → même granularité)
# _priority  : 0 pour Extended (gardé), 1 pour Account (écarté si doublon)
df_all = df_all \
    .withColumn(
        "_dedup_min",
        F.date_format(F.date_trunc("minute", F.col("listen_ts")), "yyyy-MM-dd HH:mm")
    ) \
    .withColumn(
        "_priority",
        F.when(F.col("_source") == "extended", 0).otherwise(1)
    )

# Window : pour chaque (artist, track, minute), garder la ligne de plus haute priorité
w = Window \
    .partitionBy("artistName", "trackName", "_dedup_min") \
    .orderBy("_priority")

df_merged = df_all \
    .withColumn("_rn", F.row_number().over(w)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn", "_source", "_dedup_min", "_priority")

total_merged = df_merged.count()
print(f"\nTotal après déduplication : {total_merged:,}")
print(f"Doublons supprimés        : {df_all.count() - total_merged:,}")
df_merged.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts")
).show(truncate=False)

Total avant déduplication : 390,575


+--------+------+
| _source| count|
+--------+------+
| account| 92037|
|extended|298538|
+--------+------+




Total après déduplication : 292,470


Doublons supprimés        : 98,105


+-------------------+-------------------+------------------+----------------+
|premier            |dernier            |artistes_distincts|titres_distincts|
+-------------------+-------------------+------------------+----------------+
|2020-11-21 12:47:13|2025-05-05 23:45:00|11631             |30518           |
+-------------------+-------------------+------------------+----------------+



## 5. Nettoyage & Features temporelles

Règles :
- **Filtre >30s** : `msPlayed >= 30 000` — élimine les skips et pré-écoutes
- **Features temporelles** : heure, jour, mois, année, semaine
- **`is_night`** : 22h–5h

In [6]:
# Filtre écoutes réelles (>30s)
df_streams = df_merged.filter(F.col("msPlayed") >= 30000)
print(f"Lignes après filtre >30s : {df_streams.count():,}")

# Features temporelles
df_streams = df_streams \
    .withColumn("listen_year",    F.year("listen_ts")) \
    .withColumn("listen_month",   F.date_format("listen_ts", "yyyy-MM")) \
    .withColumn("listen_hour",    F.hour("listen_ts")) \
    .withColumn("listen_weekday", F.dayofweek("listen_ts")) \
    .withColumn("listen_week",    F.weekofyear("listen_ts")) \
    .withColumn("minutes_played", F.round(F.col("msPlayed") / 60000.0, 2))

df_streams = df_streams.withColumn(
    "is_night",
    F.when((F.col("listen_hour") >= 22) | (F.col("listen_hour") <= 5), True).otherwise(False)
)

# Schema final
df_streams_final = df_streams.select(
    "artistName",
    "trackName",
    "msPlayed",
    "minutes_played",
    "trackUri",
    "skipped",
    "shuffle",
    "listen_ts",
    "listen_year",
    "listen_month",
    "listen_hour",
    "listen_weekday",
    "listen_week",
    "is_night",
)

print(f"Lignes finales : {df_streams_final.count():,}")
df_streams_final.printSchema()
df_streams_final.show(5, truncate=45)

Lignes après filtre >30s : 123,033


Lignes finales : 123,033
root
 |-- artistName: string (nullable = true)
 |-- trackName: string (nullable = true)
 |-- msPlayed: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- trackUri: string (nullable = true)
 |-- skipped: boolean (nullable = true)
 |-- shuffle: boolean (nullable = true)
 |-- listen_ts: timestamp (nullable = true)
 |-- listen_year: integer (nullable = true)
 |-- listen_month: string (nullable = true)
 |-- listen_hour: integer (nullable = true)
 |-- listen_weekday: integer (nullable = true)
 |-- listen_week: integer (nullable = true)
 |-- is_night: boolean (nullable = false)



+----------+---------------------------------------------+--------+--------------+------------------------------------+-------+-------+-------------------+-----------+------------+-----------+--------------+-----------+--------+
|artistName|                                    trackName|msPlayed|minutes_played|                            trackUri|skipped|shuffle|          listen_ts|listen_year|listen_month|listen_hour|listen_weekday|listen_week|is_night|
+----------+---------------------------------------------+--------+--------------+------------------------------------+-------+-------+-------------------+-----------+------------+-----------+--------------+-----------+--------+
|    $Mavro|                                     So so So|  114706|          1.91|spotify:track:3WCjKvUyEND8eOuVlJ0lkc|  false|   true|2024-04-25 21:14:52|       2024|     2024-04|         21|             5|         17|   false|
|       &ME|                                     Thandaza|  157950|          2.63|sp

## 6. Exploration

In [7]:
print("=== Période couverte ===")
df_streams_final.agg(
    F.min("listen_ts").alias("premier"),
    F.max("listen_ts").alias("dernier"),
    F.countDistinct("artistName").alias("artistes_distincts"),
    F.countDistinct("trackName").alias("titres_distincts"),
    F.count(F.when(F.col("trackUri").isNotNull(), 1)).alias("avec_uri")
).show(truncate=False)

print("\n=== Écoutes par année ===")
df_streams_final.groupBy("listen_year") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 0).alias("total_minutes")
    ) \
    .orderBy("listen_year") \
    .show()

print("\n=== Top 20 artistes ===")
df_streams_final.groupBy("artistName") \
    .agg(
        F.count("*").alias("nb_ecoutes"),
        F.round(F.sum("minutes_played"), 1).alias("total_minutes")
    ) \
    .orderBy(F.desc("nb_ecoutes")) \
    .limit(20) \
    .show(truncate=35)

print("\n=== Écoutes par heure ===")
df_streams_final.groupBy("listen_hour") \
    .count() \
    .orderBy("listen_hour") \
    .show(24)

print("\n=== % écoutes nocturnes ===")
total = df_streams_final.count()
night = df_streams_final.filter(F.col("is_night")).count()
print(f"Nuit : {night:,} / {total:,} ({night/total*100:.1f}%)")

=== Période couverte ===


+-------------------+-------------------+------------------+----------------+--------+
|premier            |dernier            |artistes_distincts|titres_distincts|avec_uri|
+-------------------+-------------------+------------------+----------------+--------+
|2020-11-21 12:47:13|2025-05-05 23:45:00|5926              |14731           |122292  |
+-------------------+-------------------+------------------+----------------+--------+


=== Écoutes par année ===


+-----------+----------+-------------+
|listen_year|nb_ecoutes|total_minutes|
+-----------+----------+-------------+
|       2020|         3|          3.0|
|       2021|       669|       1941.0|
|       2022|     29550|      77180.0|
|       2023|     39950|     104917.0|
|       2024|     41354|     115414.0|
|       2025|     11507|      30602.0|
+-----------+----------+-------------+


=== Top 20 artistes ===


+---------------+----------+-------------+
|     artistName|nb_ecoutes|total_minutes|
+---------------+----------+-------------+
|           Ziak|      1865|       4639.5|
|           Gazo|      1851|       5071.1|
|   Travis Scott|      1746|       5459.6|
|          Ninho|      1698|       4823.7|
|          Damso|      1664|       4892.6|
| menace Santana|      1594|       3537.5|
|Freeze corleone|      1443|       4292.6|
|         Josman|      1340|       3763.1|
|   Metro Boomin|       988|       2901.1|
|         Luidji|       980|       2858.9|
|           Dave|       916|       3533.9|
|         Kalash|       892|       2593.7|
|        Tiakola|       876|       2595.0|
|          Dinos|       868|       2872.5|
|          Hamza|       764|       2052.2|
|     Luv Resval|       732|       2564.7|
|          Niska|       705|       1769.6|
|  Green Montana|       689|       1448.3|
|  Zuukou mayzie|       680|       1757.0|
|            SDM|       675|       1778.6|
+----------

+-----------+-----+
|listen_hour|count|
+-----------+-----+
|          0| 4544|
|          1| 3795|
|          2| 2604|
|          3| 1566|
|          4| 1078|
|          5| 3821|
|          6| 5689|
|          7| 3840|
|          8| 4171|
|          9| 4784|
|         10| 4885|
|         11| 5036|
|         12| 5674|
|         13| 6307|
|         14| 6086|
|         15| 6926|
|         16| 7637|
|         17| 7404|
|         18| 7235|
|         19| 7026|
|         20| 6126|
|         21| 5899|
|         22| 5618|
|         23| 5282|
+-----------+-----+


=== % écoutes nocturnes ===


Nuit : 28,308 / 123,033 (23.0%)


## 7. Écriture spotify_streams (Delta, overwrite idempotent)

In [8]:
df_streams_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_STREAMS)

print(f"Ecrit : {OUT_STREAMS}")
df_check = spark.read.format("delta").load(OUT_STREAMS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

26/04/27 13:08:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Ecrit : /opt/spark/data/warehouse/spotify_streams
Verification : 123,033 lignes
+----------+------------------+--------+--------------+------------------------------------+-------+-------+-------------------+-----------+------------+-----------+--------------+-----------+--------+
|artistName|         trackName|msPlayed|minutes_played|                            trackUri|skipped|shuffle|          listen_ts|listen_year|listen_month|listen_hour|listen_weekday|listen_week|is_night|
+----------+------------------+--------+--------------+------------------------------------+-------+-------+-------------------+-----------+------------+-----------+--------------+-----------+--------+
|       &ME|          Thandaza|  299674|          4.99|spotify:track:48cPrTt0TLRsXQCLxQWwc7|  false|  false|2024-03-24 16:48:19|       2024|     2024-03|         16|             1|         12|   false|
|       &ME|          Thandaza|  361810|          6.03|spotify:track:48cPrTt0TLRsXQCLxQWwc7|  false|   true|2024

---
## 8. YourLibrary — Titres likés

Titres sauvegardés = identité musicale. La colonne `trackUri` alimente le graphe topologique (Phase 2G).

In [9]:
df_lib_raw = spark.read \
    .option("multiLine", "true") \
    .json(LIBRARY_PATH)

df_lib_raw.printSchema()

# Explosion du tableau 'tracks'
df_liked = df_lib_raw \
    .select(F.explode("tracks").alias("t")) \
    .select(
        F.col("t.artist").alias("artistName"),
        F.col("t.album").alias("albumName"),
        F.col("t.track").alias("trackName"),
        F.col("t.uri").alias("trackUri")
    ) \
    .filter(F.col("trackName").isNotNull())

print(f"Titres likés : {df_liked.count():,}")
df_liked.show(10, truncate=45)

root
 |-- albums: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album: string (nullable = true)
 |    |    |-- artist: string (nullable = true)
 |    |    |-- uri: string (nullable = true)
 |-- artists: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- uri: string (nullable = true)
 |-- bannedArtists: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- bannedTracks: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- episodes: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- other: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- shows: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- tracks: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- album: string (nullable = true)
 |    |    |-- artist: 

### Écriture spotify_liked_songs

In [10]:
df_liked.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_LIKED_SONGS)

print(f"Ecrit : {OUT_LIKED_SONGS}")
df_check = spark.read.format("delta").load(OUT_LIKED_SONGS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

Ecrit : /opt/spark/data/warehouse/spotify_liked_songs
Verification : 58 lignes
+-----------------+-------------------------------------------+--------------------------------+------------------------------------+
|       artistName|                                  albumName|                       trackName|                            trackUri|
+-----------------+-------------------------------------------+--------------------------------+------------------------------------+
|  Michael Jackson|HIStory - PAST, PRESENT AND FUTURE - BOOK I|        They Don't Care About Us|spotify:track:62qPhTdER9X8P2gEpHXWzN|
|Empire Of The Sun|                         Walking On A Dream|               We Are The People|spotify:track:5Telj6IS9PG6vn3PatM2Dm|
|    Louis Dunford|                                  The Angel|The Angel - North London Forever|spotify:track:230WMHL7rn1yctcsO0FtkO|
+-----------------+-------------------------------------------+--------------------------------+---------------------

---
## 9. Playlists

Tracks ajoutés manuellement = curation active.

In [11]:
df_pl_raw = spark.read \
    .option("multiLine", "true") \
    .json(PLAYLIST_PATH)

# Explosion playlists -> items -> track
df_playlists = df_pl_raw \
    .select(F.explode("playlists").alias("pl")) \
    .select(
        F.col("pl.name").alias("playlistName"),
        F.col("pl.lastModifiedDate").alias("lastModifiedDate"),
        F.explode("pl.items").alias("item")
    ) \
    .select(
        F.col("playlistName"),
        F.col("lastModifiedDate"),
        F.col("item.track.trackName").alias("trackName"),
        F.col("item.track.artistName").alias("artistName"),
        F.col("item.track.albumName").alias("albumName"),
        F.col("item.track.trackUri").alias("trackUri"),
        F.to_date(F.col("item.addedDate"), "yyyy-MM-dd").alias("addedDate")
    ) \
    .filter(F.col("trackName").isNotNull())

print(f"Tracks totaux dans les playlists : {df_playlists.count():,}")
df_playlists.groupBy("playlistName", "lastModifiedDate") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(35, truncate=50)

Tracks totaux dans les playlists : 6,464
+--------------------------+----------------+-----+
|              playlistName|lastModifiedDate|count|
+--------------------------+----------------+-----+
|          Fast&Furious 🏎️|      2025-05-05|  834|
|              kala kora 🌍|      2025-05-02|  423|
|                  PGE 2025|      2025-05-05|  419|
|                  Rhéto 🎓|      2025-04-25|  356|
|                   Vibe 🌊|      2025-05-04|  325|
|                Guapman 💷|      2025-04-30|  323|
|                  Whiné 🍑|      2025-04-17|  297|
|                    Ski 🎿|      2025-05-05|  275|
|                Rooftop 🏠|      2025-05-04|  265|
|                 Angels 🦋|      2025-05-03|  239|
|               Vacation 🎒|      2025-05-04|  237|
|           SpaceCookies 🍪|      2025-05-05|  170|
|               Lovin'it ⭐️|      2025-05-03|  169|
|Qué Calor 🇪🇸🇨🇴🇩🇴🇲🇽|      2025-05-05|  157|
|                Therapy 🥥|      2025-05-03|  145|
|             Adventure 🏔️|      2025-05-05

### Écriture spotify_playlists

In [12]:
df_playlists.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_PLAYLISTS)

print(f"Ecrit : {OUT_PLAYLISTS}")
df_check = spark.read.format("delta").load(OUT_PLAYLISTS)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=45)

Ecrit : /opt/spark/data/warehouse/spotify_playlists
Verification : 6,464 lignes
+------------+----------------+------------+----------+--------------------+------------------------------------+----------+
|playlistName|lastModifiedDate|   trackName|artistName|           albumName|                            trackUri| addedDate|
+------------+----------------+------------+----------+--------------------+------------------------------------+----------+
|   LaZone 🪐|      2025-05-05|      BOKeTE| Bad Bunny|DeBÍ TiRAR MáS FOToS|spotify:track:79x7xtoCLpf6l32Zz6mWo4|2025-01-09|
|   LaZone 🪐|      2025-05-05|        DtMF| Bad Bunny|DeBÍ TiRAR MáS FOToS|spotify:track:3sK8wGT43QFpWrvNQsrQya|2025-01-09|
|   LaZone 🪐|      2025-05-05|CAFé CON RON| Bad Bunny|DeBÍ TiRAR MáS FOToS|spotify:track:6VNXmo59yDYgcwLS17UNAW|2025-01-10|
+------------+----------------+------------+----------+--------------------+------------------------------------+----------+
only showing top 3 rows



---
## 10. Recherches — spotify_searches

2 483 requêtes de recherche depuis `SearchQueries.json`.  
Format : `{ platform, searchTime, searchQuery, searchInteractionURIs }`

In [13]:
df_search_raw = spark.read \
    .option("multiLine", "true") \
    .json(SEARCH_PATH)

print(f"Recherches brutes : {df_search_raw.count():,}")
df_search_raw.printSchema()
df_search_raw.show(5, truncate=60)

# Normalisation
# searchTime format : "2025-02-07T11:07:53.687Z[UTC]" — strip [UTC] suffix
df_searches = df_search_raw \
    .filter(F.col("searchQuery").isNotNull() & (F.length(F.col("searchQuery")) > 0)) \
    .select(
        F.to_timestamp(
            F.regexp_replace(F.col("searchTime"), r"\[UTC\]$", ""),
            "yyyy-MM-dd'T'HH:mm:ss.SSSX"
        ).alias("search_ts"),
        F.trim(F.col("searchQuery")).alias("query"),
        F.col("platform")
    ) \
    .filter(F.col("search_ts").isNotNull()) \
    .withColumn("event_hour",    F.hour("search_ts")) \
    .withColumn("event_weekday", F.dayofweek("search_ts"))

print(f"\nRecherches nettoyées : {df_searches.count():,}")
df_searches.agg(
    F.min("search_ts").alias("premier"),
    F.max("search_ts").alias("dernier")
).show(truncate=False)
df_searches.show(10, truncate=55)

Recherches brutes : 2,483
root
 |-- platform: string (nullable = true)
 |-- searchInteractionURIs: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- searchQuery: string (nullable = true)
 |-- searchTime: string (nullable = true)

+--------+---------------------+-----------+-----------------------------+
|platform|searchInteractionURIs|searchQuery|                   searchTime|
+--------+---------------------+-----------+-----------------------------+
|        |                   []|          u|2025-02-07T11:07:53.687Z[UTC]|
|        |                   []|         uo|2025-02-07T11:07:53.690Z[UTC]|
|        |                   []|         uo|2025-02-07T11:07:53.752Z[UTC]|
|        |                   []|          u|2025-02-07T11:07:53.755Z[UTC]|
|        |                   []|         uo|2025-02-07T11:07:53.959Z[UTC]|
+--------+---------------------+-----------+-----------------------------+
only showing top 5 rows


Recherches nettoyées : 2,483
+------------

### Écriture spotify_searches

In [14]:
df_searches.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(OUT_SEARCHES)

print(f"Ecrit : {OUT_SEARCHES}")
df_check = spark.read.format("delta").load(OUT_SEARCHES)
print(f"Verification : {df_check.count():,} lignes")
df_check.show(3, truncate=55)

Ecrit : /opt/spark/data/warehouse/spotify_searches
Verification : 2,483 lignes
+-----------------------+-----+--------+----------+-------------+
|              search_ts|query|platform|event_hour|event_weekday|
+-----------------------+-----+--------+----------+-------------+
|2025-02-07 11:07:53.687|    u|        |        11|            6|
| 2025-02-07 11:07:53.69|   uo|        |        11|            6|
|2025-02-07 11:07:53.752|   uo|        |        11|            6|
+-----------------------+-----+--------+----------+-------------+
only showing top 3 rows



---
## Récapitulatif

In [15]:
tables = [
    ("spotify_streams",    OUT_STREAMS),
    ("spotify_liked_songs", OUT_LIKED_SONGS),
    ("spotify_playlists",  OUT_PLAYLISTS),
    ("spotify_searches",   OUT_SEARCHES),
]

print("\n" + "="*55)
print("  Tables Spotify ecrites dans le warehouse")
print("="*55)
for name, path in tables:
    try:
        n = spark.read.format("delta").load(path).count()
        print(f"  {name:<25} {n:>8,} lignes")
    except Exception as e:
        print(f"  {name:<25} ERREUR: {e}")
print("="*55 + "\n")


  Tables Spotify ecrites dans le warehouse
  spotify_streams            123,033 lignes
  spotify_liked_songs             58 lignes
  spotify_playlists            6,464 lignes
  spotify_searches             2,483 lignes



In [16]:
spark.stop()